# 3D Calibration Surfaces

Two interactive 3D surfaces comparing market-implied probability vs historical outcome frequency.

- **Surface 1 (Implied):** what the market thinks (token price)
- **Surface 2 (Realized):** what actually happens (observed Up frequency)

Axes: time remaining (300 to 0s), BTC price variation (%), probability (0 to 1).

In [66]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

## Load data

In [67]:
TIME_STEP = 5
KEEP_SECONDS = set(range(0, 300, TIME_STEP))

ts = pd.read_parquet("../data/processed/btc_5m_timeseries.parquet",
                     columns=["second", "implied_prob", "btc_pct_change", "winner_binary"])
ts = ts[ts["second"].isin(KEEP_SECONDS)]
ts = ts.dropna(subset=["implied_prob", "btc_pct_change"])
ts = ts[ts["winner_binary"].isin([0, 1])].copy()
ts["time_remaining"] = 300 - ts["second"]
ts.drop(columns=["second"], inplace=True)

print(f"Rows after sampling every {TIME_STEP}s: {len(ts):,}")

Rows after sampling every 5s: 550,358


## Build the two surfaces

For each (time_remaining, btc_pct_change_bin) cell, compute:
- **Implied surface:** mean of `implied_prob`
- **Realized surface:** mean of `winner_binary` (actual Up frequency)

In [68]:
N_BTC_BINS = 30
MIN_COUNT = 20

_, bin_edges = pd.qcut(ts["btc_pct_change"], q=N_BTC_BINS, retbins=True, duplicates="drop")
ts["btc_bin_idx"] = np.searchsorted(bin_edges[1:-1], ts["btc_pct_change"].values)
bin_mids = 0.5 * (bin_edges[:-1] + bin_edges[1:])
ts["btc_mid"] = bin_mids[ts["btc_bin_idx"]]

btc_mid_to_idx = {m: i for i, m in enumerate(sorted(set(bin_mids)))}
ts["btc_y"] = ts["btc_mid"].map(btc_mid_to_idx)

grid = ts.groupby(["time_remaining", "btc_y"]).agg(
    implied_prob=("implied_prob", "mean"),
    implied_std=("implied_prob", "std"),
    realized_prob=("winner_binary", "mean"),
    realized_std=("winner_binary", "std"),
    btc_mid=("btc_mid", "first"),
    count=("winner_binary", "count"),
).reset_index()
grid = grid[grid["count"] >= MIN_COUNT]
grid["implied_std"] = grid["implied_std"].fillna(0)
grid["realized_std"] = grid["realized_std"].fillna(0)

btc_tickvals = sorted(btc_mid_to_idx.values())
btc_ticktext = [f"{m:.3f}%" for m, _ in sorted(btc_mid_to_idx.items(), key=lambda x: x[1])]

del ts
print(f"Grid cells: {len(grid):,}")

Grid cells: 1,756


In [69]:
Z_implied_pivot = grid.pivot_table(index="btc_y", columns="time_remaining", values="implied_prob")
Z_realized_pivot = grid.pivot_table(index="btc_y", columns="time_remaining", values="realized_prob")
Z_impl_std = grid.pivot_table(index="btc_y", columns="time_remaining", values="implied_std")
Z_real_std = grid.pivot_table(index="btc_y", columns="time_remaining", values="realized_std")

X = Z_implied_pivot.columns.values.astype(float)
Y = Z_implied_pivot.index.values.astype(float)
Z_implied = Z_implied_pivot.values
Z_realized = Z_realized_pivot.values
Z_impl_hi = np.clip(Z_implied + Z_impl_std.values, 0, 1)
Z_impl_lo = np.clip(Z_implied - Z_impl_std.values, 0, 1)
Z_real_hi = np.clip(Z_realized + Z_real_std.values, 0, 1)
Z_real_lo = np.clip(Z_realized - Z_real_std.values, 0, 1)

print(f"Surface grid: {len(X)} x {len(Y)} (time x btc_bins)")

Surface grid: 60 x 30 (time x btc_bins)


In [70]:
# Variance surfaces prepared (mean +/- 1 std)
print("Variance bands ready")

Variance bands ready


## Surface 1: Market-implied probability

In [71]:
fig1 = go.Figure()

fig1.add_trace(go.Surface(
    x=X, y=Y, z=Z_impl_hi,
    colorscale=[[0, "rgba(200,200,200,0.3)"], [1, "rgba(200,200,200,0.3)"]],
    showscale=False, connectgaps=True, opacity=0.3, name="+1 std", hoverinfo="skip",
))
fig1.add_trace(go.Surface(
    x=X, y=Y, z=Z_impl_lo,
    colorscale=[[0, "rgba(200,200,200,0.3)"], [1, "rgba(200,200,200,0.3)"]],
    showscale=False, connectgaps=True, opacity=0.3, name="-1 std", hoverinfo="skip",
))
fig1.add_trace(go.Surface(
    x=X, y=Y, z=Z_implied,
    colorscale="RdYlGn",
    cmin=0, cmax=1,
    colorbar=dict(title="Implied prob."),
    connectgaps=True,
    opacity=0.9,
    name="Mean",
))

fig1.update_layout(
    title="Surface 1: Market-implied probability (with +/- 1 std variance band)",
    scene=dict(
        xaxis_title="Time remaining (s)",
        yaxis_title="BTC price change (%)",
        yaxis=dict(tickvals=btc_tickvals[::5], ticktext=btc_ticktext[::5]),
        zaxis_title="Implied probability",
        zaxis=dict(range=[0, 1]),
    ),
    autosize=True,
)
fig1.write_html("../docs/surface_implied.html", auto_open=True)
print("Opened in browser: docs/surface_implied.html")

Opened in browser: docs/surface_implied.html


## Surface 2: Realized outcome frequency

In [56]:
fig2 = go.Figure(data=[go.Surface(
    x=X, y=Y, z=Z_realized,
    colorscale="RdYlGn",
    cmin=0, cmax=1,
    colorbar=dict(title="Realized Up rate"),
    connectgaps=True,
)])

fig2.update_layout(
    title="Surface 2: Realized outcome frequency",
    scene=dict(
        xaxis_title="Time remaining (s)",
        yaxis_title="BTC price change (%)",
        yaxis=dict(tickvals=btc_tickvals[::5], ticktext=btc_ticktext[::5]),
        zaxis_title="Observed Up frequency",
        zaxis=dict(range=[0, 1]),
    ),
    autosize=True,
)
fig2.write_html("../docs/surface_realized.html", auto_open=True)
print("Opened in browser: docs/surface_realized.html")

Opened in browser: docs/surface_realized.html


## Both surfaces overlaid

In [57]:
fig3 = go.Figure()

fig3.add_trace(go.Surface(
    x=X, y=Y, z=Z_implied,
    colorscale="Blues",
    cmin=0, cmax=1,
    opacity=0.7,
    name="Implied",
    showscale=False,
    connectgaps=True,
))

fig3.add_trace(go.Surface(
    x=X, y=Y, z=Z_realized,
    colorscale="Reds",
    cmin=0, cmax=1,
    opacity=0.7,
    name="Realized",
    showscale=False,
    connectgaps=True,
))

fig3.update_layout(
    title="Implied (blue) vs Realized (red) probability surfaces",
    scene=dict(
        xaxis_title="Time remaining (s)",
        yaxis_title="BTC price change (%)",
        yaxis=dict(tickvals=btc_tickvals[::5], ticktext=btc_ticktext[::5]),
        zaxis_title="Probability",
        zaxis=dict(range=[0, 1]),
    ),
    autosize=True,
)
fig3.write_html("../docs/surface_overlay.html", auto_open=True)
print("Opened in browser: docs/surface_overlay.html")

Opened in browser: docs/surface_overlay.html


## Calibration error surface (Implied - Realized)

Red = market overestimates Up probability, Blue = market underestimates.

In [72]:
Z_error = Z_implied - Z_realized
max_abs = np.nanmax(np.abs(Z_error))

fig4 = go.Figure(data=[go.Surface(
    x=X, y=Y, z=Z_error,
    colorscale="RdBu_r",
    cmin=-max_abs, cmax=max_abs,
    colorbar=dict(title="Error (Impl. - Real.)"),
    connectgaps=True,
)])

fig4.update_layout(
    title="Calibration error surface (Implied - Realized)",
    scene=dict(
        xaxis_title="Time remaining (s)",
        yaxis_title="BTC price change (%)",
        yaxis=dict(tickvals=btc_tickvals[::5], ticktext=btc_ticktext[::5]),
        zaxis_title="Calibration error",
    ),
    autosize=True,
)
fig4.write_html("../docs/surface_error.html", auto_open=True)
print("Opened in browser: docs/surface_error.html")

Opened in browser: docs/surface_error.html


## Calibration gap volume

Volume between the implied and realized surfaces, split by sign. Green = market underestimates Up (realized > implied). Red = market overestimates Up (implied > realized).

In [73]:
max_abs = np.nanmax(np.abs(Z_error))

fig5 = go.Figure(data=[go.Surface(
    x=X, y=Y, z=Z_error,
    surfacecolor=Z_error,
    colorscale="RdBu_r",
    cmin=-max_abs, cmax=max_abs,
    colorbar=dict(title="Implied - Realized"),
    connectgaps=True,
)])

fig5.update_layout(
    title="Calibration gap (red = overestimates Up, white = calibrated, blue = underestimates)",
    scene=dict(
        xaxis_title="Time remaining (s)",
        yaxis_title="BTC price change (%)",
        yaxis=dict(tickvals=btc_tickvals[::5], ticktext=btc_ticktext[::5]),
        zaxis_title="Calibration error (implied - realized)",
    ),
    autosize=True,
)
fig5.write_html("../docs/surface_gap_volume.html", auto_open=True)
print("Opened in browser: docs/surface_gap_volume.html")

Opened in browser: docs/surface_gap_volume.html
